# Snippet from Math-Constraints-Duality-and-Shadow-Prices.md


In [ ]:
import numpy as np
from scipy.optimize import minimize, LinearConstraint
from typing import Tuple, Dict, Optional

class ShadowPriceEstimator:
    """
    Approximate shadow prices through finite differences.
    
    Attributes:
        epsilon: Perturbation for derivatives.
        max_perturbations: Limit on constraints.
        clip_threshold: Bound extreme values.
    """
    
    def __init__(
        self,
        epsilon: float = 1e-4,
        max_perturbations: int = 100,
        clip_threshold: float = 1e3
    ):
        self.epsilon = epsilon
        self.max_perturbations = max_perturbations
        self.clip_threshold = clip_threshold
    
    def estimate(
        self,
        objective_func,
        A: np.ndarray,
        b: np.ndarray,
        x_incumbent: np.ndarray,
        U_incumbent: float
    ) -> Dict[str, any]:
        """
        Estimate shadows for A x ≤ b.
        
        Args:
            objective_func: U(x) to maximize.
            A: m x n matrix.
            b: m bounds.
            x_incumbent: Current optimum.
            U_incumbent: Current utility.
        
        Returns:
            Shadows, bindings, variances.
        """
        m = len(b)
        shadows = np.zeros(m)
        binding = []
        variances = np.zeros(m)
        
        slack = b - A @ x_incumbent
        binding_tol = 1e-6
        
        for i in range(min(m, self.max_perturbations)):
            if slack[i] < binding_tol:
                binding.append(i)
            
            b_perturbed = b.copy()
            b_perturbed[i] += self.epsilon
            
            try:
                constraint = LinearConstraint(
                    A,
                    -np.inf * np.ones(m),
                    b_perturbed
                )
                
                result = minimize(
                    lambda x: -objective_func(x),
                    x0=x_incumbent,
                    method='trust-constr',
                    constraints=constraint,
                    options={'maxiter': 100, 'verbose': 0}
                )
                
                if result.success:
                    U_perturbed = -result.fun
                    delta_U = U_perturbed - U_incumbent
                    shadows[i] = delta_U / self.epsilon
                    
                    b_double = b.copy()
                    b_double[i] += 2 * self.epsilon
                    result2 = minimize(
                        lambda x: -objective_func(x),
                        x0=result.x,
                        method='trust-constr',
                        constraints=LinearConstraint(A, -np.inf, b_double),
                        options={'maxiter': 50}
                    )
                    if result2.success:
                        delta_U2 = (-result2.fun - U_incumbent) / (2 * self.epsilon)
                        variances[i] = abs(shadows[i] - delta_U2)
            
            except Exception:
                shadows[i] = 0.0
                variances[i] = np.inf
        
        shadows = np.clip(shadows, -self.clip_threshold, self.clip_threshold)
        high_variance_flags = variances > 0.2
        
        return {
            'shadow_prices': shadows.tolist(),
            'binding_indices': binding,
            'variances': variances.tolist(),
            'high_variance_flags': high_variance_flags.tolist(),
            'slack': slack.tolist(),
            'metadata': {
                'epsilon': self.epsilon,
                'num_constraints': m,
                'num_binding': len(binding),
                'max_shadow': float(np.max(np.abs(shadows)))
            }
        }

def project_feasible(
    x: np.ndarray,
    A: np.ndarray,
    b: np.ndarray,
    max_iter: int = 1000,
    eta: float = 0.01
) -> Tuple[np.ndarray, bool]:
    """
    Project to feasible set using projected gradient descent.
    
    Args:
        x: Start point.
        A: Constraints.
        b: Bounds.
        max_iter: Iterations.
        eta: Step size.
    
    Returns:
        Projected x, convergence flag.
    """
    x_proj = x.copy()
    
    for _ in range(max_iter):
        violations = A @ x_proj - b
        max_violation = np.max(violations)
        
        if max_violation <= 0:
            return x_proj, True
        
        grad = A.T @ np.maximum(violations, 0)
        x_proj = x_proj - eta * grad
        x_proj = np.maximum(x_proj, 0)
    
    return x_proj, False

# Example
if __name__ == "__main__":
    def utility(x):
        return -0.5 * np.sum(x**2) + np.sum(x)
    
    A = np.array([
        [1.0, 2.0],
        [1.0, 1.0]
    ])
    b = np.array([3.0, 2.0])
    
    x0 = np.array([0.5, 0.5])
    x_feas, converged = project_feasible(x0, A, b)
    
    if converged:
        print(f"Feasible point: {x_feas}")
        
        result = minimize(
            lambda x: -utility(x),
            x0=x_feas,
            method='trust-constr',
            constraints=LinearConstraint(A, -np.inf, b)
        )
        
        if result.success:
            x_opt = result.x
            U_opt = -result.fun
            print(f"Optimal x: {x_opt}, U: {U_opt:.4f}")
            
            estimator = ShadowPriceEstimator(epsilon=1e-4)
            shadows = estimator.estimate(utility, A, b, x_opt, U_opt)
            
            print(f"\nShadow Prices: {shadows['shadow_prices']}")
            print(f"Binding: {shadows['binding_indices']}")
            print(f"Max shadow: {shadows['metadata']['max_shadow']:.4f}")
            
            if shadows['metadata']['max_shadow'] > 0.3:
                print("High-value constraint found.")
                print(f"Relax constraint {np.argmax(shadows['shadow_prices'])}")
